In [8]:
import pandas as pd
import re

# Load data
df = pd.read_csv("tanner_report.csv")

patterns = {
    # Remote File Inclusion (RFI) - URLs not from the same host
    "RFI": r"(?:https?|ftp):\/\/(?!202\.10\.35\.215)([\w.-]+)(?:\/|$)",

    # Local File Inclusion (LFI) - traversals or sensitive files
    "LFI": r"(\.\.\/)+|\/(etc|proc|var|home|root|dev|tmp)\/",

    # Cross-Site Scripting (XSS)
    "XSS": r"(?i)(<script.*?>.*?</script>|<.*?on\w+\s*=.*?>|javascript:|%3Cscript|alert\s*\(|document\.cookie|<\/script>)",

    # SQL Injection
    "SQLi": r"(?i)(\bselect\b.*?\bfrom\b|\binsert\b.*?\binto\b|\bunion\b.*?\bselect\b|drop\s+table|--|\bOR\b\s+\d+=\d+|sleep\s*\(|benchmark\s*\(|\bupdate\b.*?\bset\b)",

    # Command Injection
    "Command Injection": r"(;|\|\||&&|\b(cat|ls|wget|curl|whoami|ping|nc|bash|sh|python|perl)\b)",

    # PHP Code Injection
    "PHP Code Injection": r"(?i)(eval\s*\(|assert\s*\(|system\s*\(|passthru\s*\(|base64_decode\s*\(|shell_exec\s*\(|`[^`]*`)",

    # PHP Object Injection (POI)
    "PHP Object Injection": r"O:\d+:\"[^\"]+\":\d+:\{.*?\}",

    # CRLF Injection
    "CRLF": r"(%0d%0a|\r\n|\n\r|\r|\n)",

    # XML External Entity (XXE)
    "XXE": r"(?i)<\?xml.*?\?>|<!DOCTYPE\s+[a-zA-Z]+.*?>",

    # Template Injection (e.g. Jinja, Twig)
    "Template Injection": r"(\{\{.*?\}\}|\{%.+?%\}|__proto__|constructor\s*\(|toString\s*\()"
}


def detect_attack(path, detection_name=None):
    # If detection_name is provided and matches "sqli" or "xss", use that
    if detection_name is not None:
        if detection_name.lower() == "sqli":
            return "SQLi"
        elif detection_name.lower() == "xss":
            return "XSS"
    for attack_type, pattern in patterns.items():
        if re.search(pattern, path, re.IGNORECASE):
            return attack_type
    return "Unknown"

d_attack = {
    "Unknown": 0,
    "RFI": 1,
    "LFI": 2,
    "XSS": 3,
    "SQLi": 4,
    "Command Injection": 5,
    "PHP Code Injection": 6,
    "PHP Object Injection": 7,
    "CRLF": 8,
    "XXE": 9,
    "Template Injection": 10
}

df["attack_name"] = df.apply(lambda row: detect_attack(row["path"], row["detection_name"]), axis=1)
df["attack_type"] = df["attack_name"].map(d_attack)

# df["attack_name"] = df["path"].apply(detect_attack)
# df["attack_type"] = df["attack_name"].map(d_attack)

In [9]:
df.to_csv('detected.csv')